In [9]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
data_path = Path(os.environ["DATA_PATH"])

In [97]:
df_tree_inventory = (
    pd.read_csv(data_path / "urban_tree_inventory.csv")
    .rename(columns={"Families and species": "name"})
    .assign(
        name=lambda df: df["name"].str.casefold(),
        name_num_words=lambda df: df["name"].str.split(" ").str.len(),
    )
    .query("name_num_words >= 2")
    .drop(columns=["name_num_words"])
    .replace({0: False, 1: True})
    .filter(regex="name|Region")
)

df_mun = (
    pd.read_csv(data_path / "IPAB_species.csv", true_values=["X"])
    .replace(np.nan, False)
    .rename(columns={"Nome científico": "name"})
    .assign(
        name=lambda df: df["name"].str.casefold(),
        in_inventory=lambda df: df["name"].isin(df_tree_inventory["name"]),
    )
    .merge(df_tree_inventory[["name", "Region_SE"]], on="name", how="left")
    .rename(columns={"Region_SE": "in_southeast"})
    .assign(
        in_southeast=lambda df: df["in_southeast"]
        .replace(0, False)
        .replace(1, True)
        .fillna(False),
    )
)

C:\Users\lain\AppData\Local\Temp\ipykernel_29600\122645916.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({0: False, 1: True})
C:\Users\lain\AppData\Local\Temp\ipykernel_29600\122645916.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(np.nan, False)
C:\Users\lain\AppData\Local\Temp\ipykernel_29600\122645916.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the futu

In [102]:
df_tree_inventory

,name,Region_S,Region_SE,Region_N,Region_NE,Region_CW
1,sambucus australis,False,False,False,False,True
2,sambucus nigra,True,True,True,False,False
4,liquidambar styraciflua,True,False,False,False,False
6,anacardium occidentale,False,True,True,True,True
7,astronium fraxinifolium,False,False,True,False,False
...,...,...,...,...,...,...
539,citharexylum myrianthum,True,False,False,False,False
540,duranta vestita,False,True,False,False,False
542,qualea grandiflora,False,False,True,False,False
543,qualea parviflora,False,False,True,False,True


In [105]:
df_mun

,name,Nome Comum,Porte,Cerrado,Mata Atlântica,Mata ciliar,in_inventory,in_southeast
0,albizia niopoides,Farinha-seca,grande,False,True,True,False,False
1,anadenanthera macrocarpa,Angico-vermelho,grande,True,True,False,False,False
2,andira fraxinifolia,Angelim-doce,grande,False,True,False,True,False
3,apeiba tibourbou,Pente-de-macaco,médio,False,True,True,True,False
4,aspidosperma pyrifolium,Guatambu,médio,True,False,True,False,False
...,...,...,...,...,...,...,...,...
82,tabebuia roseoalba,Ipê-branco,médio,True,True,False,True,True
83,tapirira guianensis,Peito-de-pombo,grande,True,True,True,True,False
84,terminalia argentea,Capitão-do-campo,grande,True,True,True,True,False
85,tipuana tipu,Tipuana,grande,False,True,False,True,True


In [114]:
with pd.ExcelWriter("inventarios.xlsx") as writer:
    df_mun.assign(
        name=lambda df: df["name"].str.capitalize(),
        Porte=lambda df: df["Porte"].replace(
            {"grande": "Grande", "médio": "Mediano", "pequeno": "Pequeño"},
        ),
    ).rename(
        columns={
            "name": "Especie",
            "Nome Comum": "Nombre Común",
            "Porte": "Tamaño",
            "Mata Atlântica": "Bosque Atlántico",
            "Mata ciliar": "Bosque ribereño",
            "in_inventory": "En inventario",
            "in_southeast": "En sureste",
        },
    ).replace({True: "Sí", False: "No"}).to_excel(writer, sheet_name="1", index=False)

    df_tree_inventory.assign(name=lambda df: df["name"].str.capitalize()).sort_values(
        "name",
    ).rename(
        columns={
            "name": "Especie",
            "Region_S": "Región Sur",
            "Region_SE": "Región Sureste",
            "Region_N": "Región Norte",
            "Region_NE": "Región Noreste",
            "Region_CW": "Región Central-Oeste",
        },
    ).replace({True: "Sí", False: "No"}).to_excel(writer, sheet_name="2", index=False)